<a href="https://colab.research.google.com/github/550tealeaves/DATA-70500-working-with-data/blob/main/NY_Sports_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GOAL - The most important skill of Week 5: every task has TWO versions — one in SQL and one in Python. You'll write both and compare the results. By the end of this week you'll be able to use either tool interchangeably and know when each one is the right choice.

## 1. Load dataset

In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# ── YANKEES BATTING (2026, through ~15 games) ───────────────────
yank_bat = [
  # name,               pos,  avg,   obp,   slg,   ops,   hr, rbi, g
  ('Ben Rice',          '1B',  .342,  .421,  .589,  1.010, 4,  12,  15),
  ('Giancarlo Stanton', 'DH',  .326,  .398,  .543,  .941,  1,  7,   13),
  ('Aaron Judge',       'CF',  .224,  .341,  .449,  .790,  3,  7,   15),
  ('Cody Bellinger',    'RF',  .222,  .301,  .389,  .690,  1,  5,   14),
  ('Amed Rosario',      'SS',  .289,  .318,  .422,  .740,  2,  6,   15),
  ('Oswaldo Cabrera',   '3B',  .241,  .298,  .362,  .660,  1,  4,   13),
  ('Austin Wells',      'C',   .198,  .276,  .341,  .617,  1,  3,   12),
  ('Trent Grisham',     'LF',  .211,  .295,  .316,  .611,  0,  2,   11),
]
bat_cols = ['name','pos','avg','obp','slg','ops','hr','rbi','games']
df_yank_bat = pd.DataFrame(yank_bat, columns=bat_cols)

# ── YANKEES PITCHING (2026 starters + relievers) ────────────────
yank_pit = [
  # name,              role,      w, l, era,  whip, ip,   k,  bb
  ('Max Fried',        'Starter', 2, 0, 1.35, 0.90, 26.2, 28, 5),
  ('Cam Schlittler',   'Starter', 2, 1, 1.62, 1.01, 22.1, 24, 7),
  ('Ryan Weathers',    'Starter', 1, 1, 2.81, 1.18, 16.0, 14, 8),
  ('Will Warren',      'Starter', 1, 1, 3.07, 1.22, 14.2, 13, 6),
  ('Jake Bird',        'Reliever',1, 0, 1.80, 0.95, 10.0, 12, 3),
  ('Fernando Cruz',    'Reliever',1, 0, 2.35, 1.10,  7.2, 9,  2),
  ('Clay Holmes',      'Closer',  0, 0, 3.12, 1.28,  8.2, 11, 4),
]
pit_cols = ['name','role','wins','losses','era','whip','ip','k','bb']
df_yank_pit = pd.DataFrame(yank_pit, columns=pit_cols)

# ── KNICKS PLAYERS (2025-26 season averages) ────────────────────
knicks_players = [
  # name,                 pos, ppg,  rpg, apg, spg, bpg, fg_pct, min
  ('Jalen Brunson',       'G',  26.0, 3.3, 6.8, 0.9, 0.2, 0.481, 34.9),
  ('Karl-Anthony Towns',  'C',  20.1, 11.9,3.0, 0.9, 0.5, 0.499, 31.1),
  ('OG Anunoby',          'F',  17.0, 5.3, 2.2, 1.6, 0.7, 0.486, 33.0),
  ('Mikal Bridges',       'F',  14.7, 4.1, 3.9, 1.3, 0.8, 0.488, 33.3),
  ('Josh Hart',           'F',   8.1, 7.5, 4.9, 1.1, 0.2, 0.506, 29.8),
  ('Miles McBride',       'G',  12.9, 2.8, 3.1, 0.8, 0.1, 0.451, 26.4),
  ('Mitchell Robinson',   'C',   4.2, 8.8, 0.8, 0.3, 1.2, 0.621, 18.9),
  ('Jose Alvarado',       'G',   7.8, 2.1, 3.7, 1.0, 0.2, 0.441, 22.1),
]
kp_cols = ['name','pos','ppg','rpg','apg','spg','bpg','fg_pct','mpg']
df_knicks = pd.DataFrame(knicks_players, columns=kp_cols)

# ── KNICKS LAST 10 GAMES ────────────────────────────────────────
knicks_games = [
  # opponent,           pts_for, pts_against, home_away, win
  ('Raptors',            112, 95,  'Home', 1),
  ('Celtics',            108, 104, 'Away', 1),
  ('76ers',               99, 91,  'Home', 1),
  ('Heat',               101, 88,  'Away', 1),
  ('Bucks',               97, 103, 'Home', 0),
  ('Hawks',              115, 107, 'Away', 1),
  ('Hornets',             88,  92, 'Home', 0),
  ('Cavaliers',          104,  98, 'Away', 1),
  ('Magic',              109,  97, 'Home', 1),
  ('Nets',               121,  88, 'Home', 1),
]
kg_cols = ['opponent','pts_for','pts_against','home_away','win']
df_knicks_games = pd.DataFrame(knicks_games, columns=kg_cols)

# ── LOAD ALL INTO SQLITE ────────────────────────────────────────
conn = sqlite3.connect(':memory:')
df_yank_bat.to_sql('yankees_batting',  conn, index=False, if_exists='replace')
df_yank_pit.to_sql('yankees_pitching', conn, index=False, if_exists='replace')
df_knicks.to_sql('knicks_players',     conn, index=False, if_exists='replace')
df_knicks_games.to_sql('knicks_games', conn, index=False, if_exists='replace')
print('All four tables loaded!')

All four tables loaded!


In [3]:
print(yank_bat, '\n')
print(yank_pit, '\n')
print(knicks_players, '\n')
print(knicks_games, '\n')

[('Ben Rice', '1B', 0.342, 0.421, 0.589, 1.01, 4, 12, 15), ('Giancarlo Stanton', 'DH', 0.326, 0.398, 0.543, 0.941, 1, 7, 13), ('Aaron Judge', 'CF', 0.224, 0.341, 0.449, 0.79, 3, 7, 15), ('Cody Bellinger', 'RF', 0.222, 0.301, 0.389, 0.69, 1, 5, 14), ('Amed Rosario', 'SS', 0.289, 0.318, 0.422, 0.74, 2, 6, 15), ('Oswaldo Cabrera', '3B', 0.241, 0.298, 0.362, 0.66, 1, 4, 13), ('Austin Wells', 'C', 0.198, 0.276, 0.341, 0.617, 1, 3, 12), ('Trent Grisham', 'LF', 0.211, 0.295, 0.316, 0.611, 0, 2, 11)] 

[('Max Fried', 'Starter', 2, 0, 1.35, 0.9, 26.2, 28, 5), ('Cam Schlittler', 'Starter', 2, 1, 1.62, 1.01, 22.1, 24, 7), ('Ryan Weathers', 'Starter', 1, 1, 2.81, 1.18, 16.0, 14, 8), ('Will Warren', 'Starter', 1, 1, 3.07, 1.22, 14.2, 13, 6), ('Jake Bird', 'Reliever', 1, 0, 1.8, 0.95, 10.0, 12, 3), ('Fernando Cruz', 'Reliever', 1, 0, 2.35, 1.1, 7.2, 9, 2), ('Clay Holmes', 'Closer', 0, 0, 3.12, 1.28, 8.2, 11, 4)] 

[('Jalen Brunson', 'G', 26.0, 3.3, 6.8, 0.9, 0.2, 0.481, 34.9), ('Karl-Anthony Towns',

## Download CSV file

In [4]:
# Convert to CSV & download
df_yank_bat.to_csv('yank_bat.csv', index=False)
df_yank_pit.to_csv('yank_pit.csv', index=False)
df_knicks.to_csv('knicks_players.csv', index=False)
df_knicks_games.to_csv('knicks_games.csv', index=False)
from google.colab import files
files.download('yank_bat.csv')
files.download('yank_pit.csv')
files.download('knicks_players.csv')
files.download('knicks_games.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Write the SQL version first — run it and check the output. Then write the Python version to match the same result.
- Add a comment at the end of each task with one sentence: 'SQL was better here because...' or 'Python was easier here because...' This reflection is the most important part.

## Show all Yankees batters with OPS above 0.700

In [2]:
# -- SQL version
pd.read_sql_query('''
    SELECT name, pos, avg, ops
    FROM yankees_batting
    WHERE ops > 0.700
    ORDER BY ops DESC
''', conn)

,name,pos,avg,ops
0,Ben Rice,1B,0.342,1.010
1,Giancarlo Stanton,DH,0.326,0.941
2,Aaron Judge,CF,0.224,0.790
3,Amed Rosario,SS,0.289,0.740


In [4]:
# Python version
df_yank_bat[
    df_yank_bat['ops'] > 0.700
][['name','pos','avg','ops']].sort_values('ops',
    ascending=False)

,name,pos,avg,ops
0,Ben Rice,1B,0.342,1.010
1,Giancarlo Stanton,DH,0.326,0.941
2,Aaron Judge,CF,0.224,0.790
4,Amed Rosario,SS,0.289,0.740


## Filter & Sort — Find the Yankees' power hitters
- Find all Yankees batters with 2 or more home runs this season. Show their **name, position, home runs, RBI, and OPS**. Sort by home runs descending.

In [5]:
# -- SQL version
pd.read_sql_query('''
    SELECT name, pos, hr, rbi, ops
    FROM yankees_batting
    WHERE hr >= 2
    ORDER BY hr DESC
''', conn)

,name,pos,hr,rbi,ops
0,Ben Rice,1B,4,12,1.01
1,Aaron Judge,CF,3,7,0.79
2,Amed Rosario,SS,2,6,0.74


- SQL was much easier to use

In [9]:
# -- Python version
df_yank_bat[
    df_yank_bat['hr'] >= 2
][['name', 'pos', 'hr', 'rbi', 'ops']].sort_values('hr', ascending=False)

,name,pos,hr,rbi,ops
0,Ben Rice,1B,4,12,1.01
2,Aaron Judge,CF,3,7,0.79
4,Amed Rosario,SS,2,6,0.74


Python seems understandable but I have a tough time rememberin the differnt functions

## Aggregation — Knicks starters vs bench comparison
- Create a new column called 'role' that labels players as 'Starter' if they average 30+ minutes, or 'Bench' if under 30 minutes. Then calculate the average PPG, RPG, and FG% for each role group. Which group is more efficient?

In [15]:
#
pd.read_sql_query('''
SELECT name, mpg,
  CASE
    WHEN mpg >= 30 THEN 'Starter'
    ELSE 'Bench'
      END AS 'Role'
  FROM knicks_players
''', conn)

,name,mpg,Role
0,Jalen Brunson,34.9,Starter
1,Karl-Anthony Towns,31.1,Starter
2,OG Anunoby,33.0,Starter
3,Mikal Bridges,33.3,Starter
4,Josh Hart,29.8,Bench
5,Miles McBride,26.4,Bench
6,Mitchell Robinson,18.9,Bench
7,Jose Alvarado,22.1,Bench


In [19]:
pd.read_sql_query('''
SELECT
  CASE
    WHEN mpg >= 30 THEN 'Starter'
    ELSE 'Bench'
  END AS role,
  AVG(ppg) AS 'avg ppg',
  AVG(rpg) AS 'avg rpg',
  AVG(fg_pct) AS 'avg fg%'
FROM knicks_players
GROUP BY role;
''', conn)

,role,avg ppg,avg rpg,avg fg%
0,Bench,8.25,5.30,0.50475
1,Starter,19.45,6.15,0.48850
